In [ ]:
!pip install rdkit

In [ ]:
!pip install PyTDC --no-deps

In [ ]:
from tdc.single_pred import Tox
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

data = Tox(name='LD50_Zhu')
df = data.get_data()
print(df.shape)
print(df.head())

In [ ]:
# Random split
random_split = data.get_split(method='random')
train_r, val_r, test_r = random_split['train'], random_split['valid'], random_split['test']

# Scaffold split
scaffold_split = data.get_split(method='scaffold')
train_s, val_s, test_s = scaffold_split['train'], scaffold_split['valid'], scaffold_split['test']

print(f"Random  — Train: {len(train_r)}, Val: {len(val_r)}, Test: {len(test_r)}")
print(f"Scaffold — Train: {len(train_s)}, Val: {len(val_s)}, Test: {len(test_s)}")

# Morgan Fingerprint Featurization

In [ ]:
def smiles_to_morgan(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    return np.array(fp)

def featurize(df):
    X, y, valid_idx = [], [], []
    for i, row in df.iterrows():
        fp = smiles_to_morgan(row['Drug'])
        if fp is not None:
            X.append(fp)
            y.append(row['Y'])
            valid_idx.append(i)
    return np.array(X), np.array(y)

# Featurize all splits
X_train_r, y_train_r = featurize(train_r)
X_val_r,   y_val_r   = featurize(val_r)
X_test_r,  y_test_r  = featurize(test_r)

X_train_s, y_train_s = featurize(train_s)
X_val_s,   y_val_s   = featurize(val_s)
X_test_s,  y_test_s  = featurize(test_s)

print(f"Train shape: {X_train_r.shape}, Labels: {y_train_r.shape}")

# EDA

In [ ]:
import matplotlib.pyplot as plt

# Check label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y_train_r, bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('LD50 Distribution (Random Split - Train)')
axes[0].set_xlabel('LD50 (log mol/kg)')
axes[0].set_ylabel('Count')

axes[1].hist(y_train_s, bins=50, color='darkorange', edgecolor='black')
axes[1].set_title('LD50 Distribution (Scaffold Split - Train)')
axes[1].set_xlabel('LD50 (log mol/kg)')

plt.tight_layout()
plt.show()

# Basic stats
print("Random Split — Train Label Stats:")
print(f"  Mean: {y_train_r.mean():.3f}, Std: {y_train_r.std():.3f}, Min: {y_train_r.min():.3f}, Max: {y_train_r.max():.3f}")

print("\nScaffold Split — Train Label Stats:")
print(f"  Mean: {y_train_s.mean():.3f}, Std: {y_train_s.std():.3f}, Min: {y_train_s.min():.3f}, Max: {y_train_s.max():.3f}")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
y_train_r_scaled = scaler.fit_transform(y_train_r.reshape(-1,1)).flatten()
y_val_r_scaled   = scaler.transform(y_val_r.reshape(-1,1)).flatten()
y_test_r_scaled  = scaler.transform(y_test_r.reshape(-1,1)).flatten()

scaler_s = StandardScaler()
y_train_s_scaled = scaler_s.fit_transform(y_train_s.reshape(-1,1)).flatten()
y_val_s_scaled   = scaler_s.transform(y_val_s.reshape(-1,1)).flatten()
y_test_s_scaled  = scaler_s.transform(y_test_s.reshape(-1,1)).flatten()

print("Done! Ready for models.")

# Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

def evaluate(y_true, y_pred, label=""):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    sp   = spearmanr(y_true, y_pred).correlation
    print(f"[{label}] MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f} | Spearman: {sp:.4f}")
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'Spearman': sp}

# Train on random split
lr_r = LinearRegression()
lr_r.fit(X_train_r, y_train_r_scaled)

# Train on scaffold split
lr_s = LinearRegression()
lr_s.fit(X_train_s, y_train_s_scaled)

# Evaluate
print("=== Linear Regression ===")
lr_r_val  = evaluate(y_val_r_scaled,  lr_r.predict(X_val_r),  "Random - Val")
lr_r_test = evaluate(y_test_r_scaled, lr_r.predict(X_test_r), "Random - Test")
lr_s_val  = evaluate(y_val_s_scaled,  lr_s.predict(X_val_s),  "Scaffold - Val")
lr_s_test = evaluate(y_test_s_scaled, lr_s.predict(X_test_s), "Scaffold - Test")

# Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

# Hyperparameter tuning for alpha
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
ridge_cv_r = GridSearchCV(Ridge(), param_grid={'alpha': alphas},
                          cv=5, scoring='neg_mean_squared_error')
ridge_cv_r.fit(X_train_r, y_train_r_scaled)

ridge_cv_s = GridSearchCV(Ridge(), param_grid={'alpha': alphas},
                          cv=5, scoring='neg_mean_squared_error')
ridge_cv_s.fit(X_train_s, y_train_s_scaled)

print(f"Best alpha (Random):   {ridge_cv_r.best_params_}")
print(f"Best alpha (Scaffold): {ridge_cv_s.best_params_}")

# Evaluate
print("\n=== Ridge Regression ===")
rr_r_val  = evaluate(y_val_r_scaled,  ridge_cv_r.predict(X_val_r),  "Random - Val")
rr_r_test = evaluate(y_test_r_scaled, ridge_cv_r.predict(X_test_r), "Random - Test")
rr_s_val  = evaluate(y_val_s_scaled,  ridge_cv_s.predict(X_val_s),  "Scaffold - Val")
rr_s_test = evaluate(y_test_s_scaled, ridge_cv_s.predict(X_test_s), "Scaffold - Test")

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_r = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_r.fit(X_train_r, y_train_r_scaled)

rf_s = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_s.fit(X_train_s, y_train_s_scaled)

print("=== Random Forest (Default) ===")
rf_r_val  = evaluate(y_val_r_scaled,  rf_r.predict(X_val_r),  "Random - Val")
rf_r_test = evaluate(y_test_r_scaled, rf_r.predict(X_test_r), "Random - Test")
rf_s_val  = evaluate(y_val_s_scaled,  rf_s.predict(X_val_s),  "Scaffold - Val")
rf_s_test = evaluate(y_test_s_scaled, rf_s.predict(X_test_s), "Scaffold - Test")

# Random Forest Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', 0.3]
}

rf_cv_r = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_grid,
    n_iter=20, cv=5,
    scoring='neg_mean_squared_error',
    random_state=42, n_jobs=-1, verbose=1
)
rf_cv_r.fit(X_train_r, y_train_r_scaled)

rf_cv_s = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_grid,
    n_iter=20, cv=5,
    scoring='neg_mean_squared_error',
    random_state=42, n_jobs=-1, verbose=1
)
rf_cv_s.fit(X_train_s, y_train_s_scaled)

print(f"Best params (Random):   {rf_cv_r.best_params_}")
print(f"Best params (Scaffold): {rf_cv_s.best_params_}")

print("\n=== Random Forest (Tuned) ===")
rf_r_val  = evaluate(y_val_r_scaled,  rf_cv_r.predict(X_val_r),  "Random - Val")
rf_r_test = evaluate(y_test_r_scaled, rf_cv_r.predict(X_test_r), "Random - Test")
rf_s_val  = evaluate(y_val_s_scaled,  rf_cv_s.predict(X_val_s),  "Scaffold - Val")
rf_s_test = evaluate(y_test_s_scaled, rf_cv_s.predict(X_test_s), "Scaffold - Test")

# DNN

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Convert to tensors
def to_tensor(X, y):
    return TensorDataset(
        torch.FloatTensor(X).to(device),
        torch.FloatTensor(y).to(device)
    )

train_ds_r = to_tensor(X_train_r, y_train_r_scaled)
val_ds_r   = to_tensor(X_val_r,   y_val_r_scaled)
test_ds_r  = to_tensor(X_test_r,  y_test_r_scaled)

train_ds_s = to_tensor(X_train_s, y_train_s_scaled)
val_ds_s   = to_tensor(X_val_s,   y_val_s_scaled)
test_ds_s  = to_tensor(X_test_s,  y_test_s_scaled)

# Define DNN
class ToxDNN(nn.Module):
    def __init__(self, input_dim=2048, hidden_dims=[512, 256, 128], dropout=0.3):
        super(ToxDNN, self).__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)

print("Architecture ready!")

#  DNN Training Loop

In [ ]:
def train_dnn_v2(train_ds, val_ds, epochs=200, batch_size=128, lr=1e-3, dropout=0.5):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)

    model = ToxDNN(hidden_dims=[512, 256, 128], dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_model_state = None
    train_losses, val_losses = [], []
    patience_counter = 0
    early_stop_patience = 20

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(X_batch)
        train_loss /= len(train_ds)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                val_loss += criterion(model(X_batch), y_batch).item() * len(X_batch)
        val_loss /= len(val_ds)

        scheduler.step(val_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Patience: {patience_counter}/{early_stop_patience}")

        if patience_counter >= early_stop_patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_model_state)
    return model, train_losses, val_losses

print("=== Training on Random Split (v2) ===")
model_r, tl_r, vl_r = train_dnn_v2(train_ds_r, val_ds_r)

print("\n=== Training on Scaffold Split (v2) ===")
model_s, tl_s, vl_s = train_dnn_v2(train_ds_s, val_ds_s)

In [ ]:
def evaluate_dnn(model, dataset):
    loader = DataLoader(dataset, batch_size=256)
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds.append(model(X_batch).cpu().numpy())
            targets.append(y_batch.cpu().numpy())
    return np.concatenate(preds), np.concatenate(targets)

# Random split
pred_val_r,  true_val_r  = evaluate_dnn(model_r, val_ds_r)
pred_test_r, true_test_r = evaluate_dnn(model_r, test_ds_r)

# Scaffold split
pred_val_s,  true_val_s  = evaluate_dnn(model_s, val_ds_s)
pred_test_s, true_test_s = evaluate_dnn(model_s, test_ds_s)

print("=== DNN Results ===")
dnn_r_val  = evaluate(true_val_r,  pred_val_r,  "Random - Val")
dnn_r_test = evaluate(true_test_r, pred_test_r, "Random - Test")
dnn_s_val  = evaluate(true_val_s,  pred_val_s,  "Scaffold - Val")
dnn_s_test = evaluate(true_test_s, pred_test_s, "Scaffold - Test")

In [ ]:
# Get RF predictions on clipped splits
rf_pred_val_r  = rf_cv_r.predict(X_val_r)
rf_pred_test_r = rf_cv_r.predict(X_test_r)
rf_pred_val_s  = rf_cv_s.predict(X_val_s)
rf_pred_test_s = rf_cv_s.predict(X_test_s)

# Try both weightings
for w_dnn, w_rf in [(0.5, 0.5), (0.7, 0.3)]:
    print(f"\n=== Ensemble (DNN {w_dnn} / RF {w_rf}) ===")
    
    ens_val_r  = w_dnn*pred_val_r  + w_rf*rf_pred_val_r
    ens_test_r = w_dnn*pred_test_r + w_rf*rf_pred_test_r
    ens_val_s  = w_dnn*pred_val_s  + w_rf*rf_pred_val_s
    ens_test_s = w_dnn*pred_test_s + w_rf*rf_pred_test_s

    evaluate(true_val_r,  ens_val_r,  "Random - Val")
    evaluate(true_test_r, ens_test_r, "Random - Test")
    evaluate(true_val_s,  ens_val_s,  "Scaffold - Val")
    evaluate(true_test_s, ens_test_s, "Scaffold - Test")

# MPNN (Message Passing Neural Network)

In [ ]:
import torch

# Get torch version for correct install
torch_version = torch.__version__.split('+')[0]
cuda_version = 'cu121'  # Colab default

print(f"Torch version: {torch_version}")

!pip install torch-geometric
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-${torch_version}+${cuda_version}.html

SMILES to Graph Converter

In [ ]:
import torch
from torch_geometric.data import Data, DataLoader as GeoDataLoader
from rdkit import Chem
import numpy as np

def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetNumRadicalElectrons(),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing()),
        atom.GetTotalNumHs(),
        atom.GetNumImplicitHs(),
    ]

def bond_features(bond):
    bt = bond.GetBondTypeAsDouble()
    return [
        bt,
        int(bond.GetIsConjugated()),
        int(bond.GetIsAromatic()),
        int(bond.IsInRing()),
    ]

def smiles_to_graph(smiles, y=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Node features
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], 
                     dtype=torch.float)

    # Edge indices + features
    edge_idx, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_idx += [[i, j], [j, i]]
        edge_attr += [bf, bf]

    edge_index = torch.tensor(edge_idx, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if y is not None:
        data.y = torch.tensor([y], dtype=torch.float)
    return data

def build_graph_dataset(df, y_scaled):
    graphs = []
    for (_, row), y in zip(df.iterrows(), y_scaled):
        g = smiles_to_graph(row['Drug'], y)
        if g is not None:
            graphs.append(g)
    return graphs

print("Building graph datasets...")
train_graphs_r = build_graph_dataset(train_r, y_train_r_scaled)
val_graphs_r   = build_graph_dataset(val_r,   y_val_r_scaled)
test_graphs_r  = build_graph_dataset(test_r,  y_test_r_scaled)

train_graphs_s = build_graph_dataset(train_s, y_train_s_scaled)
val_graphs_s   = build_graph_dataset(val_s,   y_val_s_scaled)
test_graphs_s  = build_graph_dataset(test_s,  y_test_s_scaled)

print(f"Train graphs: {len(train_graphs_r)}")
print(f"Sample graph — Nodes: {train_graphs_r[0].x.shape}, Edges: {train_graphs_r[0].edge_index.shape}")

MPNN Model

In [ ]:
from torch_geometric.nn import NNConv, global_mean_pool, global_max_pool
import torch.nn.functional as F

class MPNN(nn.Module):
    def __init__(self, node_dim=8, edge_dim=4, hidden_dim=128, n_layers=3, dropout=0.3):
        super().__init__()
        
        # Edge networks for message passing
        self.edge_nets = nn.ModuleList()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        in_dim = node_dim
        for _ in range(n_layers):
            edge_net = nn.Sequential(
                nn.Linear(edge_dim, 32),
                nn.ReLU(),
                nn.Linear(32, in_dim * hidden_dim)
            )
            self.edge_nets.append(edge_net)
            self.convs.append(NNConv(in_dim, hidden_dim, edge_net, aggr='mean'))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
            in_dim = hidden_dim

        self.dropout = dropout
        
        # Readout head
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),  # *2 for mean+max pooling
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = \
            data.x, data.edge_index, data.edge_attr, data.batch

        # Message passing layers
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Graph readout — concat mean and max pooling
        x = torch.cat([
            global_mean_pool(x, batch),
            global_max_pool(x, batch)
        ], dim=1)

        return self.head(x).squeeze(1)

# Test forward pass
sample_loader = GeoDataLoader(train_graphs_r[:4], batch_size=4)
sample_batch  = next(iter(sample_loader))
model_test    = MPNN().to(device)
out = model_test(sample_batch.to(device))
print(f"MPNN output shape: {out.shape}")
print("MPNN architecture ready!")

 MPNN Training Loop

In [ ]:
from torch_geometric.data import DataLoader as GeoDataLoader

def train_mpnn(train_graphs, val_graphs, epochs=300, batch_size=64, 
               lr=1e-3, dropout=0.3):
    train_loader = GeoDataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    val_loader   = GeoDataLoader(val_graphs,   batch_size=batch_size)

    model = MPNN(dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, patience=15, factor=0.5)  # removed verbose
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_model_state = None
    train_losses, val_losses = [], []
    patience_counter = 0
    early_stop_patience = 50

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            pred = model(batch)
            loss = criterion(pred, batch.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * batch.num_graphs
        train_loss /= len(train_graphs)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                val_loss += criterion(model(batch), batch.y).item() * batch.num_graphs
        val_loss /= len(val_graphs)

        scheduler.step(val_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Patience: {patience_counter}/{early_stop_patience}")

        if patience_counter >= early_stop_patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_model_state)
    return model, train_losses, val_losses

print("=== Training MPNN (Random Split) ===")
mpnn_r, tl_mpnn_r, vl_mpnn_r = train_mpnn(train_graphs_r, val_graphs_r)

print("\n=== Training MPNN (Scaffold Split) ===")
mpnn_s, tl_mpnn_s, vl_mpnn_s = train_mpnn(train_graphs_s, val_graphs_s)

In [ ]:
def evaluate_mpnn(model, graphs):
    loader = GeoDataLoader(graphs, batch_size=256)
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            preds.append(model(batch).cpu().numpy())
            targets.append(batch.y.cpu().numpy())
    return np.concatenate(preds), np.concatenate(targets)

# Evaluate
pred_val_mpnn_r,  true_val_mpnn_r  = evaluate_mpnn(mpnn_r, val_graphs_r)
pred_test_mpnn_r, true_test_mpnn_r = evaluate_mpnn(mpnn_r, test_graphs_r)
pred_val_mpnn_s,  true_val_mpnn_s  = evaluate_mpnn(mpnn_s, val_graphs_s)
pred_test_mpnn_s, true_test_mpnn_s = evaluate_mpnn(mpnn_s, test_graphs_s)

print("=== MPNN Results ===")
evaluate(true_val_mpnn_r,  pred_val_mpnn_r,  "Random - Val")
evaluate(true_test_mpnn_r, pred_test_mpnn_r, "Random - Test")
evaluate(true_val_mpnn_s,  pred_val_mpnn_s,  "Scaffold - Val")
evaluate(true_test_mpnn_s, pred_test_mpnn_s, "Scaffold - Test")

# ChemBERTa 

In [ ]:
!pip install transformers datasets

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Load pretrained ChemBERTa tokenizer and model
model_name = "seyonec/ChemBERTa-zinc-base-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
chemberta_base = AutoModel.from_pretrained(model_name)

print(f"ChemBERTa loaded!")
print(f"Hidden size: {chemberta_base.config.hidden_size}")

# Dataset class
class SMILESDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            smiles_list,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

# Build datasets
print("Tokenizing SMILES...")
train_smiles_r = train_r['Drug'].tolist()
val_smiles_r   = val_r['Drug'].tolist()
test_smiles_r  = test_r['Drug'].tolist()

train_smiles_s = train_s['Drug'].tolist()
val_smiles_s   = val_s['Drug'].tolist()
test_smiles_s  = test_s['Drug'].tolist()

train_chem_r = SMILESDataset(train_smiles_r, y_train_r_scaled, tokenizer)
val_chem_r   = SMILESDataset(val_smiles_r,   y_val_r_scaled,   tokenizer)
test_chem_r  = SMILESDataset(test_smiles_r,  y_test_r_scaled,  tokenizer)

train_chem_s = SMILESDataset(train_smiles_s, y_train_s_scaled, tokenizer)
val_chem_s   = SMILESDataset(val_smiles_s,   y_val_s_scaled,   tokenizer)
test_chem_s  = SMILESDataset(test_smiles_s,  y_test_s_scaled,  tokenizer)

print("Done!")

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

class ChemBERTaRegressor(nn.Module):
    def __init__(self, base_model, hidden_size=768, dropout=0.3):
        super().__init__()
        self.bert = base_model
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Use [CLS] token representation
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.regressor(cls_output).squeeze(1)


def train_chemberta(train_ds, val_ds, epochs=50, batch_size=16, lr=2e-5, dropout=0.3):  # batch 32 -> 16
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size)

    base = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
    model = ChemBERTaRegressor(base, dropout=dropout).to(device)

    optimizer = torch.optim.AdamW([
        {'params': model.bert.parameters(),      'lr': 2e-5},
        {'params': model.regressor.parameters(), 'lr': 1e-3}
    ], weight_decay=1e-2)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    early_stop_patience = 10

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast():  # mixed precision — saves memory
                preds = model(input_ids, attention_mask)
                loss  = criterion(preds, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * len(labels)
        train_loss /= len(train_ds)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels         = batch['labels'].to(device)
                with torch.cuda.amp.autocast():
                    val_loss += criterion(model(input_ids, attention_mask), labels).item() * len(labels)
        val_loss /= len(val_ds)

        scheduler.step()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        print(f"Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Patience: {patience_counter}/{early_stop_patience}")

        if patience_counter >= early_stop_patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
    return model

print("=== Fine-tuning ChemBERTa (Random Split) ===")
chemberta_r = train_chemberta(train_chem_r, val_chem_r)

print("\n=== Fine-tuning ChemBERTa (Scaffold Split) ===")
chemberta_s = train_chemberta(train_chem_s, val_chem_s)

In [ ]:
def evaluate_chemberta(model, dataset):
    loader = DataLoader(dataset, batch_size=16)
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            with torch.cuda.amp.autocast():
                out = model(input_ids, attention_mask)
            preds.append(out.cpu().numpy())
            targets.append(batch['labels'].numpy())
    return np.concatenate(preds), np.concatenate(targets)

pred_val_cb_r,  true_val_cb_r  = evaluate_chemberta(chemberta_r, val_chem_r)
pred_test_cb_r, true_test_cb_r = evaluate_chemberta(chemberta_r, test_chem_r)
pred_val_cb_s,  true_val_cb_s  = evaluate_chemberta(chemberta_s, val_chem_s)
pred_test_cb_s, true_test_cb_s = evaluate_chemberta(chemberta_s, test_chem_s)

print("=== ChemBERTa Results ===")
evaluate(true_val_cb_r,  pred_val_cb_r,  "Random - Val")
evaluate(true_test_cb_r, pred_test_cb_r, "Random - Test")
evaluate(true_val_cb_s,  pred_val_cb_s,  "Scaffold - Val")
evaluate(true_test_cb_s, pred_test_cb_s, "Scaffold - Test")

In [ ]:
# Check all required variables exist
check = {
    'y_train_r_scaled': 'y_train_r_scaled' in dir(),
    'pred_test_r (DNN)': 'pred_test_r' in dir(),
    'true_test_r (DNN)': 'true_test_r' in dir(),
    'pred_test_s (DNN)': 'pred_test_s' in dir(),
    'true_test_s (DNN)': 'true_test_s' in dir(),
    'rf_cv_r': 'rf_cv_r' in dir(),
    'rf_cv_s': 'rf_cv_s' in dir(),
    'tl_r (DNN losses)': 'tl_r' in dir(),
    'vl_r (DNN losses)': 'vl_r' in dir(),
    'tl_s (DNN losses)': 'tl_s' in dir(),
    'vl_s (DNN losses)': 'vl_s' in dir(),
    'X_test_r': 'X_test_r' in dir(),
    'X_test_s': 'X_test_s' in dir(),
}

for k, v in check.items():
    print(f"{'✅' if v else '❌'} {k}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['font.size'] = 11

# ── 1. MODEL COMPARISON BAR CHART ──────────────────────────────────────────
models = ['Ridge', 'RF Tuned', 'DNN', 'Ensemble\n70/30', 'MPNN', 'ChemBERTa']

r2_random   = [0.509, 0.603, 0.625, 0.635, 0.583, 0.536]
r2_scaffold = [0.178, 0.180, 0.262, 0.271, 0.071, 0.173]
sp_random   = [0.658, 0.726, 0.738, 0.742, 0.730, 0.671]
sp_scaffold = [0.523, 0.562, 0.613, 0.625, 0.585, 0.500]

x = np.arange(len(models))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² comparison
axes[0].bar(x - width/2, r2_random,   width, label='Random Split',   color='steelblue',  alpha=0.85)
axes[0].bar(x + width/2, r2_scaffold, width, label='Scaffold Split', color='darkorange', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, fontsize=9)
axes[0].set_ylabel('R²')
axes[0].set_title('R² Comparison Across Models')
axes[0].legend()
axes[0].set_ylim(0, 0.8)
for i, (v1, v2) in enumerate(zip(r2_random, r2_scaffold)):
    axes[0].text(i - width/2, v1 + 0.01, f'{v1:.3f}', ha='center', fontsize=7)
    axes[0].text(i + width/2, v2 + 0.01, f'{v2:.3f}', ha='center', fontsize=7)

# Spearman comparison
axes[1].bar(x - width/2, sp_random,   width, label='Random Split',   color='steelblue',  alpha=0.85)
axes[1].bar(x + width/2, sp_scaffold, width, label='Scaffold Split', color='darkorange', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, fontsize=9)
axes[1].set_ylabel('Spearman Correlation')
axes[1].set_title('Spearman Correlation Across Models')
axes[1].legend()
axes[1].set_ylim(0, 0.9)
for i, (v1, v2) in enumerate(zip(sp_random, sp_scaffold)):
    axes[1].text(i - width/2, v1 + 0.01, f'{v1:.3f}', ha='center', fontsize=7)
    axes[1].text(i + width/2, v2 + 0.01, f'{v2:.3f}', ha='center', fontsize=7)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 2. DNN TRAINING LOSS CURVES ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(tl_r, label='Train Loss', color='steelblue')
axes[0].plot(vl_r, label='Val Loss',   color='darkorange')
axes[0].set_title('DNN Training — Random Split')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()

axes[1].plot(tl_s, label='Train Loss', color='steelblue')
axes[1].plot(vl_s, label='Val Loss',   color='darkorange')
axes[1].set_title('DNN Training — Scaffold Split')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('dnn_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3. PREDICTED VS ACTUAL SCATTER ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

scatter_data = [
    (true_test_r,      rf_cv_r.predict(X_test_r),  'RF — Random',   axes[0][0]),
    (true_test_r,      pred_test_r,                 'DNN — Random',  axes[0][1]),
    (true_test_r,      0.7*pred_test_r + 0.3*rf_cv_r.predict(X_test_r), 'Ensemble — Random', axes[0][2]),
    (true_test_s,      rf_cv_s.predict(X_test_s),  'RF — Scaffold', axes[1][0]),
    (true_test_s,      pred_test_s,                 'DNN — Scaffold',axes[1][1]),
    (true_test_s,      0.7*pred_test_s + 0.3*rf_cv_s.predict(X_test_s), 'Ensemble — Scaffold', axes[1][2]),
]

for true, pred, title, ax in scatter_data:
    ax.scatter(true, pred, alpha=0.4, s=15, color='steelblue')
    mn, mx = min(true.min(), pred.min()), max(true.max(), pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect fit')
    r2 = r2_score(true, pred)
    sp = spearmanr(true, pred).correlation
    ax.set_title(f'{title}\nR²={r2:.3f} | Spearman={sp:.3f}')
    ax.set_xlabel('True LD50')
    ax.set_ylabel('Predicted LD50')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 4. RANDOM FOREST FEATURE IMPORTANCE ────────────────────────────────────
importances = rf_cv_r.best_estimator_.feature_importances_
top_idx     = np.argsort(importances)[::-1][:20]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(20), importances[top_idx], color='steelblue', alpha=0.85)
ax.set_xticks(range(20))
ax.set_xticklabels([f'Bit {i}' for i in top_idx], rotation=45, ha='right', fontsize=8)
ax.set_title('Top 20 Most Important Morgan Fingerprint Bits (Random Forest)')
ax.set_ylabel('Feature Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 5. SCAFFOLD VS RANDOM GAP ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
gap = [r - s for r, s in zip(r2_random, r2_scaffold)]
colors = ['red' if g > 0.3 else 'darkorange' if g > 0.2 else 'steelblue' for g in gap]
ax.bar(models, gap, color=colors, alpha=0.85)
ax.set_title('Generalization Gap (Random R² − Scaffold R²)')
ax.set_ylabel('R² Gap')
ax.axhline(y=0.3, color='red',    linestyle='--', alpha=0.5, label='High gap (>0.3)')
ax.axhline(y=0.2, color='orange', linestyle='--', alpha=0.5, label='Moderate gap (>0.2)')
ax.legend()
plt.tight_layout()
plt.savefig('generalization_gap.png', dpi=150, bbox_inches='tight')
plt.show()

print("All visualizations saved!")